# Bag-of-Words SL

Sample script for zero-step-sampling GRPO on the bag-of-wrds dataset

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsSLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    corr=0.01,
    num_words=15,
    num_samples=50_000,
    aux_words_ratio=0.5,
    prompt_length=128,
    filter_samples_above_n_tokens=384,
    word_decay_power=1.0,
    batch_size=64,
    eval_batch_size_multiple=4,
    lr_per_token=1.25e-7,
    backbone_lr_divisor=5.0,
    pad_to_multiple=8,
    model_name="HuggingFaceTB/SmolLM2-135M",
    train_epochs=2,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sl-example/15-words_corr-0.01_len-128_pow-1.0_ar-0.5
Dataset corr target: 0.0100
Backbone lr: 2.048e-04
Head lr:     1.024e-03


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/195 [00:00<?, ?it/s]

epoch  0  train_corr_target=-0.0034  train_corr_ground_truth=0.0424  pred_norm=0.7275  val_corr_target=0.0084  val_corr_ground_truth=0.1528


sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/195 [00:00<?, ?it/s]

epoch  1  train_corr_target=0.0059  train_corr_ground_truth=0.0306  pred_norm=3.1054  val_corr_target=0.0058  val_corr_ground_truth=0.0341


## Results

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_corr_target,train_mse_target,train_beta_target,train_corr_ground_truth,train_mse_ground_truth,train_beta_ground_truth,val_corr_target,val_mse_target,val_beta_target,val_corr_ground_truth,val_mse_ground_truth,val_beta_ground_truth
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,-0.003421,1.024322,-0.018233,0.042427,0.034722,0.002265,0.008399,1.009016,0.077936,0.152782,0.011376,0.014163
1,0.005865,1.017704,0.032982,0.030623,0.031252,0.001725,0.00579,1.1267,0.015949,0.034111,0.131534,0.000939


In [5]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.292969,0.011353,-0.589844
-0.347656,0.022217,-0.474609
-0.296875,0.007019,1.15625
-0.302734,0.003128,0.112793
-0.417969,-0.008972,2.453125
